In [ ]:
#Install requirements

# %pip install "transformers==4.44.2" "tokenizers==0.19.1" -q
# %pip install "FlagEmbedding==1.3.5" -q

# %pip install pymupdf beautifulsoup4 requests mitreattack-python \
#             langchain-text-splitters numpy pandas tqdm chromadb \
#             "ollama>=0.3" "openai>=1.30" \
#             ipykernel jupyter -q

# %pip install git+https://github.com/vibrantlabsai/ragas -q

In [2]:
import os
import pickle
import numpy as np
import pandas as pd
from collections import Counter

import pymupdf
import requests
from bs4 import BeautifulSoup
from mitreattack.stix20 import MitreAttackData
from langchain_text_splitters import RecursiveCharacterTextSplitter

import chromadb
from FlagEmbedding import BGEM3FlagModel, FlagReranker

import ollama

from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory
from ragas.metrics.collections import Faithfulness, ContextUtilization, AnswerRelevancy

from tqdm.notebook import tqdm

In [ ]:
#Pull LLM and embedding models from ollama
!ollama pull qwen3:8b
!ollama pull nomic-embed-text

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling a3de86cd1c13: 100% ▕██████████████████▏ 5.2 GB                         
pulling ae370d884f10: 100% ▕██████████████████▏ 1.7 KB                         
pulling d18a5cc71b84: 100% ▕██████████████████▏  11 KB                         
pulling cff3f395ef37: 100% ▕██████████████████▏  120 B                         
pulling 05a61d37b084: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠦ pulling manifest ⠦ pulling manifest ⠧ p

In [5]:
attacks = MitreAttackData("docs/enterprise_attack.json")

# Build lookup table
technique_to_mitigations = attacks.get_all_mitigations_mitigating_all_techniques()
technique_to_software = attacks.get_all_software_using_all_techniques()

techniques = attacks.get_techniques(
    include_subtechniques=True,
    remove_revoked_deprecated=True,
)

print(f"Total techniques: {len(techniques)}")

Total techniques: 691


In [6]:
# build ATT&CK chunks
attck_chunks = []

for technique in techniques:
    mitre_id = attacks.get_attack_id(technique.id) or ""
    tactics = [t.name for t in attacks.get_tactics_by_technique(technique.id)]

    software_entries = technique_to_software.get(technique.id, [])
    mitigation_entries = technique_to_mitigations.get(technique.id, [])

    tools = [e["object"].name for e in software_entries if e["object"].type == "tool"]
    malware = [
        e["object"].name for e in software_entries if e["object"].type == "malware"
    ]
    mitigation_names = [e["object"].name for e in mitigation_entries]
    is_subtechnique = "." in mitre_id

    chunk_text = f"""MITRE ATT&CK Technique: {technique.name}
ID: {mitre_id}
Type: {"Sub-technique" if is_subtechnique else "Technique"}
Tactics: {", ".join(tactics)}

Description:
{getattr(technique, "description", "Not available").strip()}

Detection:
{getattr(technique, "x_mitre_detection", "Not documented").strip()}

Associated Tools: {", ".join(tools) if tools else "None documented"}
Associated Malware: {", ".join(malware) if malware else "None documented"}
Mitigations: {", ".join(mitigation_names) if mitigation_names else "None documented"}""".strip()

    attck_chunks.append(
        {
            "text": chunk_text,
            "metadata": {
                "source": "MITRE ATT&CK",
                "id": mitre_id,
                "name": technique.name,
                "tactics": tactics,
                "is_subtechnique": is_subtechnique,
                "tools": tools,
                "malware": malware,
                "mitigations": mitigation_names,
            },
        }
    )

print(f"ATT&CK chunks built: {len(attck_chunks)}")
sample = attck_chunks[0]
print(f"\nSample — {sample['metadata']['id']}: {sample['metadata']['name']}")
print(sample["text"][:400])

ATT&CK chunks built: 691

Sample — T1055.011: Extra Window Memory Injection
MITRE ATT&CK Technique: Extra Window Memory Injection
ID: T1055.011
Type: Sub-technique
Tactics: Privilege Escalation, Defense Evasion

Description:
Adversaries may inject malicious code into process via Extra Window Memory (EWM) in order to evade process-based defenses as well as possibly elevate privileges. EWM injection is a method of executing arbitrary code in the address space of a separate 


In [7]:
PDF_SOURCES = {
    "docs/cisa_AA24-326A.pdf": "CISA Red Team Assessment AA24-326A",
    "docs/cisa_aa24-193a_silentshield.pdf": "CISA Red Team Assessment AA24-193A SILENTSHIELD",
    "docs/cisa_aa23-059a.pdf": "CISA Red Team Assessment AA23-059A",
    "docs/nist_sp_800-115.pdf": "NIST SP 800-115 Information Security Testing",
    "docs/nist_sp_800-61r3.pdf": "NIST SP 800-61r3 Incident Response",
    "docs/nist_sp_800-53r5.pdf": "NIST SP 800-53r5 Security and Privacy Controls",
    "docs/nist_csf_2.pdf": "NIST Cybersecurity Framework 2.0",
    "docs/cert_vuln_disclosure.pdf": "CERT Guide to Coordinated Vulnerability Disclosure",
    "docs/tiber-eu.pdf": "EU Threat Intelligence-Based Ethical Red Teaming",
    "docs/cbest_tila.pdf": "CBEST Threat Intelligence-Led Assessments",
}


#Function to extract text from PDF and split into overlapping chunks.
def parse_and_chunk_pdf(path, source_name, chunk_size=800, overlap=150):
    doc = pymupdf.open(path)
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " "],
    )
    chunks = []
    for i, page in enumerate(doc):
        text = page.get_text().strip()
        if not text:
            continue
        for j, split in enumerate(splitter.split_text(text)):
            if len(split.strip()) < 50:
                continue
            chunks.append(
                {
                    "text": split.strip(),
                    "metadata": {
                        "source": source_name,
                        "page": i + 1,
                        "chunk_index": j,
                    },
                }
            )
    return chunks


pdf_chunks = []
for path, source_name in PDF_SOURCES.items():
    if not os.path.exists(path) or os.path.getsize(path) < 10_000:
        print(f"  ⚠ SKIPPED (missing or too small): {path}")
        continue
    chunks = parse_and_chunk_pdf(path, source_name)
    pdf_chunks.extend(chunks)
    print(f"  ✓ {source_name}: {len(chunks)} chunks")

print(f"\nTotal PDF chunks: {len(pdf_chunks)}")

  ✓ CISA Red Team Assessment AA24-326A: 135 chunks
  ✓ CISA Red Team Assessment AA24-193A SILENTSHIELD: 109 chunks
  ✓ CISA Red Team Assessment AA23-059A: 99 chunks
MuPDF error: format error: No default Layer config

  ✓ NIST SP 800-115 Information Security Testing: 419 chunks
  ✓ NIST SP 800-61r3 Incident Response: 177 chunks
  ✓ NIST SP 800-53r5 Security and Privacy Controls: 2649 chunks
  ✓ NIST Cybersecurity Framework 2.0: 115 chunks
  ✓ CERT Guide to Coordinated Vulnerability Disclosure: 535 chunks
  ✓ EU Threat Intelligence-Based Ethical Red Teaming: 233 chunks
  ✓ CBEST Threat Intelligence-Led Assessments: 149 chunks

Total PDF chunks: 4620


In [8]:
# --- PTES: Scrape and chunk all seven sections ---
PTES_SECTIONS = {
    "Pre-engagement Interactions": "http://www.pentest-standard.org/index.php/Pre-engagement",
    "Intelligence Gathering": "http://www.pentest-standard.org/index.php/Intelligence_Gathering",
    "Threat Modeling": "http://www.pentest-standard.org/index.php/Threat_Modeling",
    "Vulnerability Analysis": "http://www.pentest-standard.org/index.php/Vulnerability_Analysis",
    "Exploitation": "http://www.pentest-standard.org/index.php/Exploitation",
    "Post Exploitation": "http://www.pentest-standard.org/index.php/Post_Exploitation",
    "Reporting": "http://www.pentest-standard.org/index.php/Reporting",
}


def scrape_ptes_section(url, section_name):
    """Scrape a PTES MediaWiki section and return clean text."""
    try:
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
        r.raise_for_status()
    except Exception as e:
        print(f"  Failed to fetch {section_name}: {e}")
        return None

    soup = BeautifulSoup(r.text, "html.parser")
    content_div = soup.find("div", {"id": "mw-content-text"}) or soup.find("body")
    if not content_div:
        return None

    for tag in content_div.find_all(
        ["script", "style", "table", "sup", "span.editsection"]
    ):
        tag.decompose()

    lines = [l.strip() for l in content_div.get_text(separator="\n").splitlines()]
    clean_text = "\n\n".join(l for l in lines if l)
    return clean_text


def chunk_ptes_section(text, section_name, chunk_size=800, overlap=150):
    """Split a PTES section into overlapping chunks."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " "],
    )
    return [
        {
            "text": s.strip(),
            "metadata": {"source": "PTES", "section": section_name, "chunk_index": i},
        }
        for i, s in enumerate(splitter.split_text(text))
        if len(s.strip()) >= 50
    ]


ptes_chunks = []
for section_name, url in PTES_SECTIONS.items():
    print(f"Scraping: {section_name}...")
    text = scrape_ptes_section(url, section_name)
    if text:
        chunks = chunk_ptes_section(text, section_name)
        ptes_chunks.extend(chunks)
        print(f"  → {len(chunks)} chunks")

print(f"\nTotal PTES chunks: {len(ptes_chunks)}")

Scraping: Pre-engagement Interactions...
  → 72 chunks
Scraping: Intelligence Gathering...
  → 75 chunks
Scraping: Threat Modeling...
  → 35 chunks
Scraping: Vulnerability Analysis...
  → 49 chunks
Scraping: Exploitation...
  → 30 chunks
Scraping: Post Exploitation...
  → 56 chunks
Scraping: Reporting...
  → 22 chunks

Total PTES chunks: 339


In [9]:
# --- Combine all sources ---
all_chunks = attck_chunks + pdf_chunks + ptes_chunks

print("Chunks per source:")
for source, count in Counter(c["metadata"]["source"] for c in all_chunks).items():
    print(f"  {source}: {count}")
print(f"  TOTAL: {len(all_chunks)}")

Chunks per source:
  MITRE ATT&CK: 691
  CISA Red Team Assessment AA24-326A: 135
  CISA Red Team Assessment AA24-193A SILENTSHIELD: 109
  CISA Red Team Assessment AA23-059A: 99
  NIST SP 800-115 Information Security Testing: 419
  NIST SP 800-61r3 Incident Response: 177
  NIST SP 800-53r5 Security and Privacy Controls: 2649
  NIST Cybersecurity Framework 2.0: 115
  CERT Guide to Coordinated Vulnerability Disclosure: 535
  EU Threat Intelligence-Based Ethical Red Teaming: 233
  CBEST Threat Intelligence-Led Assessments: 149
  PTES: 339
  TOTAL: 5650


## Embedding

In [ ]:
# Load BGE-M3 embedding model
embedder = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
print("BGE-M3 loaded")

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

BGE-M3 loaded


In [11]:
def truncate_chunk(text, max_chars=3000):
    return text[:max_chars] if len(text) > max_chars else text


all_texts = [truncate_chunk(c["text"]) for c in all_chunks]
all_dense = []
all_sparse = []
BATCH_SIZE = 16

for start in tqdm(range(0, len(all_texts), BATCH_SIZE), desc="Embedding chunks"):
    batch = all_texts[start : start + BATCH_SIZE]

    encoded = embedder.encode(
        batch, return_dense=True, return_sparse=True, batch_size=BATCH_SIZE
    )

    all_dense.append(encoded["dense_vecs"])
    all_sparse.extend(encoded["lexical_weights"])

dense_vecs = np.vstack(all_dense)
sparse_weights = all_sparse

print(f"Dense shape:    {dense_vecs.shape}")
print(f"Sparse entries: {len(sparse_weights)}")

Embedding chunks:   0%|          | 0/354 [00:00<?, ?it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 136.46it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 21.22it/s]


Dense shape:    (5650, 1024)
Sparse entries: 5650


In [12]:
# --- Store dense vectors in ChromaDB ---
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# Delete and recreate if rebuilding from scratch
try:
    chroma_client.delete_collection("redteam_rag")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="redteam_rag", metadata={"hnsw:space": "cosine"}
)

ids = [str(i) for i in range(len(all_chunks))]
documents = [c["text"] for c in all_chunks]

# ChromaDB requires flat metadata (no nested lists)
metadatas = []
for c in all_chunks:
    meta = dict(c["metadata"])
    for k, v in meta.items():
        if isinstance(v, list):
            meta[k] = ", ".join(v) if v else ""
    metadatas.append(meta)

BATCH_SIZE = 500
for i in range(0, len(all_chunks), BATCH_SIZE):
    end = min(i + BATCH_SIZE, len(all_chunks))
    collection.add(
        ids=ids[i:end],
        documents=documents[i:end],
        embeddings=dense_vecs[i:end].tolist(),
        metadatas=metadatas[i:end],
    )
    print(f"  Stored batch {i}:{end}")

print(f"Total stored in ChromaDB: {collection.count()}")

  Stored batch 0:500
  Stored batch 500:1000
  Stored batch 1000:1500
  Stored batch 1500:2000
  Stored batch 2000:2500
  Stored batch 2500:3000
  Stored batch 3000:3500
  Stored batch 3500:4000
  Stored batch 4000:4500
  Stored batch 4500:5000
  Stored batch 5000:5500
  Stored batch 5500:5650
Total stored in ChromaDB: 5650


In [13]:
# --- Save sparse weights to disk ---
sparse_store = {str(i): sparse_weights[i] for i in range(len(all_chunks))}

with open("sparse_weights.pkl", "wb") as f:
    pickle.dump(sparse_store, f)

print("Sparse weights saved")

# Verify consistency
assert (
    collection.count() == len(sparse_store) == len(all_chunks)
), "Mismatch between stores — recheck ingestion"
print("All stores consistent")

Sparse weights saved
All stores consistent


In [14]:
# --- Reload from disk (run this cell after a kernel restart) ---
embedder = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)

chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_collection("redteam_rag")

with open("sparse_weights.pkl", "rb") as f:
    sparse_store = pickle.load(f)

print(f"Collection: {collection.count()} chunks")
print(f"Sparse:     {len(sparse_store)} entries")

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Collection: 5650 chunks
Sparse:     5650 entries


## Retrieval

In [15]:
# --- Hybrid retrieval functions ---


def dense_query(query_vec, collection, top_k=10):
    """Retrieve top-k chunks by cosine similarity."""
    results = collection.query(
        query_embeddings=query_vec.tolist(),
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )
    ids = [int(i) for i in results["ids"][0]]
    scores = [1 - d for d in results["distances"][0]]
    return ids, scores


def sparse_query(query_sparse, sparse_store, top_k=10):
    """Retrieve top-k chunks by lexical weight dot product."""
    scores = {}
    for idx, doc_sparse in sparse_store.items():
        shared = set(query_sparse.keys()) & set(doc_sparse.keys())
        scores[int(idx)] = sum(query_sparse[t] * doc_sparse[t] for t in shared)
    sorted_ids = sorted(scores, key=scores.get, reverse=True)[:top_k]
    return sorted_ids, [scores[i] for i in sorted_ids]


def reciprocal_rank_fusion(ranked_lists, k=60):
    """Combine ranked lists via RRF (Cormack et al., 2009). k=60 is the standard constant."""
    rrf_scores = {}
    for ranked_ids in ranked_lists:
        for rank, doc_id in enumerate(ranked_ids):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (k + rank + 1)
    return rrf_scores


def hybrid_retrieve(query, embedder, collection, sparse_store, all_chunks, top_k=10):
    """Encode query, run dense + sparse retrieval, fuse with RRF."""
    q_encoded = embedder.encode([query], return_dense=True, return_sparse=True)
    q_dense = q_encoded["dense_vecs"]
    q_sparse = q_encoded["lexical_weights"][0]

    dense_ids, _ = dense_query(q_dense, collection, top_k)
    sparse_ids, _ = sparse_query(q_sparse, sparse_store, top_k)

    rrf_scores = reciprocal_rank_fusion([dense_ids, sparse_ids])
    sorted_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:top_k]

    return [all_chunks[i] for i in sorted_ids], sorted_ids

In [16]:
# --- Reranker (cross-encoder, bge-reranker-v2-m3) ---
# Loaded once; freed after use to reclaim VRAM before generation
reranker = FlagReranker("BAAI/bge-reranker-v2-m3", use_fp16=True)


def rerank(query, candidates, top_k=3):
    """Score query-candidate pairs, return top-k by reranker score."""
    pairs = [[query, c["text"]] for c in candidates]
    scores = reranker.compute_score(pairs, normalize=True)
    ranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
    return [c for _, c in ranked[:top_k]], [s for s, _ in ranked[:top_k]]

## Generation

In [17]:
# --- System prompt and generation function ---
SYSTEM_PROMPT = """You are an expert red team cybersecurity advisor.
Your role is to provide accurate, actionable guidance to red team practitioners.
Answer ONLY based on the provided context.
If the context does not contain enough information, say so explicitly.
Always cite the source document (e.g. MITRE ATT&CK T1550.002, PTES, CISA)
for each piece of advice you give."""


def generate(query, context_chunks, stream=True):
    """Generate a response from Qwen3:8b using the retrieved context."""
    context = "\n\n---\n\n".join(
        [
            f"[Source: {c['metadata']['source']} | "
            f"{c['metadata'].get('id', c['metadata'].get('section', ''))}]\n"
            f"{c['text']}"
            for c in context_chunks
        ]
    )

    response_stream = ollama.chat(
        model="qwen3:8b",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"},
        ],
        options={"temperature": 0.1, "think": False},
        stream=stream,
    )

    if stream:
        full_response = ""
        for chunk in response_stream:
            token = chunk["message"]["content"]
            print(token, end="", flush=True)
            full_response += token
        print()
        return full_response
    else:
        return response_stream["message"]["content"]

## RAG

Measures:
* [Faithfulness](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/faithfulness/)

$\text{Faithfulness Score} \in (0, 1) = \frac{\text{Number of claims in the response supported by the retrieved context}}{\text{Total number of claims in the response}}$

* [Context Utilisation](https://docs.ragas.io/en/v0.1.21/concepts/metrics/context_utilization.html)

$ x \in (0, 1)$

* [Answer Relevance](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/answer_relevance/)

$ x \in (-1, 1)$

In [18]:
# --- Full RAG pipeline ---
def rag_pipeline(query, top_k_retrieve=10, top_k_rerank=3, verbose=True):
    """
    End-to-end pipeline:
    1. Hybrid retrieval (dense + sparse + RRF) → top_k_retrieve candidates
    2. Cross-encoder reranking → top_k_rerank final chunks
    3. LLM generation with retrieved context
    """
    if verbose:
        print(f"Query: {query}\n")

    # Stage 1 — Hybrid retrieval
    candidates, candidate_ids = hybrid_retrieve(
        query, embedder, collection, sparse_store, all_chunks, top_k=top_k_retrieve
    )
    if verbose:
        print(f"Retrieved {len(candidates)} candidates")

    # Stage 2 — Rerank
    final_chunks, rerank_scores = rerank(query, candidates, top_k=top_k_rerank)
    if verbose:
        print(f"Reranked to top {top_k_rerank}:")
        for i, (chunk, score) in enumerate(zip(final_chunks, rerank_scores)):
            src = chunk["metadata"]["source"]
            name = chunk["metadata"].get("id", chunk["metadata"].get("section", ""))
            print(f"  {i+1}. [{score:.3f}] {src} | {name}")

    # Stage 3 — Generate
    answer = generate(query, final_chunks, stream=verbose)
    if verbose:
        print(f"\nAnswer:\n{answer}")

    return {
        "query": query,
        "retrieved": candidates,
        "final_chunks": final_chunks,
        "rerank_scores": rerank_scores,
        "answer": answer,
    }


# Smoke test
_ = rag_pipeline(
    "What reconnaissance techniques should I use before an assumed breach engagement?"
)

Query: What reconnaissance techniques should I use before an assumed breach engagement?



You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Retrieved 10 candidates


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Reranked to top 3:
  1. [0.022] NIST SP 800-115 Information Security Testing | 
  2. [0.014] MITRE ATT&CK | T1597.002
  3. [0.014] MITRE ATT&CK | T1596.005
To conduct effective reconnaissance before an assumed breach engagement, use the following techniques, supported by the provided sources:  

1. **Public Data Collection**  
   - Gather information from public registration data, DNS server records, newsgroups, and other openly available sources to identify system names, IP addresses, operating systems, and technical contacts. This aligns with **NIST SP 800-115** practices for external testing.  

2. **Purchase Technical Data (T1597.002)**  
   - Acquire technical details about targets from reputable sources (e.g., paid scan databases) or less-reputable ones (e.g., dark web markets). This includes employee contact info, credentials, or infrastructure specifics, as documented in **MITRE ATT&CK T1597.002**.  

3. **Scan Public Databases (T1596.005)**  
   - Use tools like Shodan or othe

## RAGAS Evaluation

In [19]:
# --- Configure RAGAS with local Ollama ---
ragas_client = AsyncOpenAI(api_key="ollama", base_url="http://localhost:11434/v1")

ragas_llm = llm_factory(
    "qwen3:8b",
    client=ragas_client,
    max_tokens=4096,
    extra_body={"options": {"think": False, "num_ctx": 8192}},
)
ragas_embeddings = embedding_factory(
    "openai", model="nomic-embed-text", client=ragas_client
)


faithfulness_scorer = Faithfulness(llm=ragas_llm)
context_utilization_scorer = ContextUtilization(llm=ragas_llm)
answer_relevancy_scorer = AnswerRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)

print("RAGAS scorers initialised")

RAGAS scorers initialised


In [20]:
# --- Run all 10 practitioner queries through the full pipeline ---
QUERIES = [
    "What reconnaissance techniques should I use before an assumed breach engagement?",
    "How do I establish persistence on a Windows host without triggering EDR?",
    "What are the rules of engagement considerations for a purple team exercise?",
    "How should lateral movement be documented during a red team engagement?",
    "What MITRE ATT&CK techniques are most relevant to Active Directory attacks?",
    "How do I scope a red team engagement for a mid-sized financial organisation?",
    "What is the difference between a red team and a penetration test?",
    "How should I handle discovering a critical zero-day during a client engagement?",
    "What tools are commonly used for command and control in adversary simulation?",
    "How do I write an executive summary for a red team report?",
]

results = []
for query in tqdm(QUERIES):
    result = rag_pipeline(query)
    results.append(result)

print(f"\nAll {len(results)} queries complete")

  0%|          | 0/10 [00:00<?, ?it/s]

Query: What reconnaissance techniques should I use before an assumed breach engagement?

Retrieved 10 candidates
Reranked to top 3:
  1. [0.022] NIST SP 800-115 Information Security Testing | 
  2. [0.014] MITRE ATT&CK | T1597.002
  3. [0.014] MITRE ATT&CK | T1596.005
To conduct effective reconnaissance before an assumed breach engagement, use the following techniques, all of which are documented in the provided sources:  

1. **Purchase Technical Data (MITRE ATT&CK T1597.002)**  
   - Acquire technical information about targets from reputable or less-reputable sources (e.g., paid subscriptions, dark web markets). This may include employee contact details, credentials, or infrastructure specifics.  
   - **Purpose**: Identify vulnerabilities or opportunities for initial access (e.g., phishing, credential theft).  

2. **Scan Databases (MITRE ATT&CK T1596.005)**  
   - Query public scan databases (e.g., Shodan, Censys) to gather data on active IP addresses, open ports, server banners, a

In [21]:
# Fix max_token limit issue

MAX_CTX_CHARS = 1500
MAX_ANSWER_CHARS = 800


# --- Compute RAGAS metrics for all results ---
async def evaluate_single(query, answer, contexts):
    # Truncate each context chunk before scoring
    contexts = [c[:MAX_CTX_CHARS] for c in contexts]
    answer = answer[:MAX_ANSWER_CHARS]
    faith = await faithfulness_scorer.ascore(
        user_input=query, response=answer, retrieved_contexts=contexts
    )
    ctx_util = await context_utilization_scorer.ascore(
        user_input=query, response=answer, retrieved_contexts=contexts
    )
    relev = await answer_relevancy_scorer.ascore(user_input=query, response=answer)
    return {
        "query": query,
        "faithfulness": round(faith.value, 3),
        "context_utilization": round(ctx_util.value, 3),
        "answer_relevancy": round(relev.value, 3),
    }


async def evaluate_all(results):
    records = []
    for r in tqdm(results):
        contexts = [c["text"] for c in r["final_chunks"]]
        scores = await evaluate_single(r["query"], r["answer"], contexts)
        records.append(scores)
        print(
            f"F={scores['faithfulness']:.3f} | "
            f"CU={scores['context_utilization']:.3f} | "
            f"AR={scores['answer_relevancy']:.3f}"
        )
    return records


# Jupyter supports top-level await
eval_records = await evaluate_all(results)

  0%|          | 0/10 [00:00<?, ?it/s]

F=1.000 | CU=1.000 | AR=0.947
F=0.889 | CU=1.000 | AR=0.890
F=0.400 | CU=1.000 | AR=0.977
F=0.250 | CU=1.000 | AR=0.926
F=0.875 | CU=1.000 | AR=0.984
F=0.800 | CU=0.583 | AR=0.914
F=1.000 | CU=1.000 | AR=1.000
F=0.333 | CU=1.000 | AR=0.947
F=1.000 | CU=1.000 | AR=0.971
F=0.667 | CU=0.833 | AR=0.887


In [22]:
# --- Results table ---
METRIC_COLS = ["faithfulness", "context_utilization", "answer_relevancy"]

df = pd.DataFrame(eval_records)
df.index = [f"Q{i+1}" for i in range(len(df))]

print("=== RAGAS Evaluation Results ===")
print(df[["query"] + METRIC_COLS].to_string())

print("\n=== Mean Scores ===")
print(df[METRIC_COLS].mean().round(3).to_string())

=== RAGAS Evaluation Results ===
                                                                                query  faithfulness  context_utilization  answer_relevancy
Q1   What reconnaissance techniques should I use before an assumed breach engagement?         1.000                1.000             0.947
Q2           How do I establish persistence on a Windows host without triggering EDR?         0.889                1.000             0.890
Q3        What are the rules of engagement considerations for a purple team exercise?         0.400                1.000             0.977
Q4            How should lateral movement be documented during a red team engagement?         0.250                1.000             0.926
Q5        What MITRE ATT&CK techniques are most relevant to Active Directory attacks?         0.875                1.000             0.984
Q6       How do I scope a red team engagement for a mid-sized financial organisation?         0.800                0.583             